Url Connection & Data Cleaning

In [99]:
import io
import os
import time
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()
nasa_api = os.getenv("NASA_MAP_KEY")
source = "VIIRS_NOAA20_NRT"
day_range = 5

top_10_locations = {
  "location_1": {"name": "Amazon Rainforest", "continent": "South America", "west_lon": -80.0, "south_lat": -5.0, "east_lon": -50.0, "north_lat": 5.0},
  "location_2": {"name": "California", "continent": "North America", "west_lon": -124.5, "south_lat": 32.5, "east_lon": -114.0, "north_lat": 42.0},
  "location_3": {"name": "Siberian Taiga", "continent": "Russia", "west_lon": 60.0, "south_lat": 50.0, "east_lon": 140.0, "north_lat": 70.0},
  "location_4": {"name": "New South Wales", "continent": "Austrailia", "west_lon": 141.0, "south_lat": -37.5, "east_lon": 154.0, "north_lat": -28.0},
  "location_5": {"name": "Pantanal", "continent": "Brazil", "west_lon": -62.0, "south_lat": -22.0, "east_lon": -55.0, "north_lat": -16.0},
  "location_6": {"name": "Alberta", "continent": "Canada", "west_lon": -120.0, "south_lat": 49.0, "east_lon": -110.0, "north_lat": 60.0},
  "location_7": {"name": "Mediterranean Basin", "continent": "Southern Europe", "west_lon": -10.0, "south_lat": 30.0, "east_lon": 40.0, "north_lat": 46.0},
  "location_8": {"name": "Indonesia", "continent": "Southeast Asia", "west_lon": 95.0, "south_lat": -11.0, "east_lon": 141.0, "north_lat": 6.0},
  "location_9": {"name": "Greece", "continent": "Southern Europe", "west_lon": 19.0, "south_lat": 34.0, "east_lon": 28.0, "north_lat": 42.0},
  "location_10": {"name": "Algeria", "continent": "North Africa", "west_lon": -9.0, "south_lat": 19.0, "east_lon": 12.0, "north_lat": 37.0}
}

dataframe_list = []
master_df = None

for location_key, location_data in top_10_locations.items():
  west_lon = location_data['west_lon']
  south_lat = location_data['south_lat']
  east_lon = location_data['east_lon']
  north_lat = location_data['north_lat']
  name = location_data['name']

  data_base_url = "https://firms.modaps.eosdis.nasa.gov/api/area"
  prediction_url = f"{data_base_url}/csv/{nasa_api}/{source}/{west_lon},{south_lat},{east_lon},{north_lat}/{day_range}"

  try: 
    response = requests.get(prediction_url)
    response.raise_for_status() 
    
    if "No data available" in response.text or len(response.text.strip()) == 0:
      print(f"No active fires found in {name} for this time range.")
      continue
    
    temp_df = pd.read_csv(io.StringIO(response.text))
    temp_df["location_name"] = name

    dataframe_list.append(temp_df)
    print(f"Successfully added {len(temp_df)} fire points from {name}.")

  except requests.exceptions.HTTPError as e:
    print(f"Error fetching {name}: {e}")
  
  time.sleep(0.5)

if dataframe_list:
    master_df = pd.concat(dataframe_list, ignore_index=True)
    master_df = master_df.drop_duplicates(subset=['latitude', 'longitude', 'bright_ti4', 'acq_date', 'acq_time'])
    print("\nHigh-capacity local aggregation complete!")
    print(f"Combined Dataset Matrix Size: {len(master_df)} Unique Operational Fire Observations")
else:
    print("\nDataset compilation failed. Check your network or API permissions.")
    raise SystemExit

Successfully added 94 fire points from Amazon Rainforest.
Successfully added 208 fire points from California.
Successfully added 2835 fire points from Siberian Taiga.
Successfully added 267 fire points from New South Wales.
Successfully added 1468 fire points from Pantanal.
Successfully added 138 fire points from Alberta.
Successfully added 2852 fire points from Mediterranean Basin.
Successfully added 1491 fire points from Indonesia.
Successfully added 141 fire points from Greece.
Successfully added 1428 fire points from Algeria.

High-capacity local aggregation complete!
Combined Dataset Matrix Size: 9718 Unique Operational Fire Observations


Data Cleaning

In [100]:
master_df.isnull().values.any() # No null values
master_df.duplicated().any() # No duplicated values
master_df.isna().sum() # No na values

latitude         0
longitude        0
bright_ti4       0
scan             0
track            0
acq_date         0
acq_time         0
satellite        0
instrument       0
confidence       0
version          0
bright_ti5       0
frp              0
daynight         0
location_name    0
dtype: int64

Data Exploration

In [101]:
variance_profile = master_df.groupby('location_name')[['frp', 'bright_ti5', 'bright_ti4']].agg(['mean','std'])
print("\n")
print(variance_profile)
print("\n")
prediction_features = master_df[['frp', 'bright_ti5', 'bright_ti4', 'track', 'scan']]
print(prediction_features.head())



                          frp             bright_ti5             bright_ti4  \
                         mean        std        mean        std        mean   
location_name                                                                 
Alberta              6.839928  13.064263  289.597029  11.101858  326.322899   
Algeria              2.744329   3.791625  292.418932   9.261658  317.222986   
Amazon Rainforest    4.662660   2.562684  290.409894   3.869538  334.629468   
California           4.223558  10.402042  292.526346  12.137486  315.605385   
Indonesia            7.048478  12.763210  291.899591   6.915788  329.742220   
Mediterranean Basin  3.900806   8.759803  294.143065   9.890317  319.754208   
New South Wales      3.694869   5.247631  283.212472   5.534497  316.666854   
Pantanal             3.871213   5.850493  291.675388   5.394010  321.167827   
Siberian Taiga       7.812857  12.843014  284.727189  10.868634  325.805005   

                                
                

Creating Risk Level Column

In [102]:
def assign_risk_level(frp_value):
  if frp_value < 5.0:
    return 0
  elif frp_value < 15.0:
    return 1
  elif frp_value < 40.0:
    return 2
  else:
    return 3

master_df['risk_level'] = master_df['frp'].apply(assign_risk_level)

master_df['is_daytime'] = master_df['daynight'].map({'D': 1, 'N': 0})

if master_df['confidence'].dtype == 'O':
    master_df['confidence_num'] = master_df['confidence'].map({'low': 0, 'nominal': 1, 'high': 2})
else:
    master_df['confidence_num'] = master_df['confidence']

master_df['is_daytime'] = master_df['is_daytime'].fillna(0).astype(int)
master_df['confidence_num'] = pd.to_numeric(master_df['confidence_num'], errors='coerce').fillna(0).astype(float)

Prediction: Fire Radiative Power & Bright Fire Spots

In [103]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

X = master_df[['track', 'scan', 'bright_ti5', 'bright_ti4', 'latitude', 'longitude', 'is_daytime', 'confidence_num']]
y = master_df['risk_level']

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size = 0.8, random_state = 42)

# Printing the shapes
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}, y_train: {y_train.shape}, y_test: {y_test.shape}")


# Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=120, criterion="log_loss", random_state=42)
rf_model.fit(X_train, y_train, sample_weight=None)
rf_predictions = rf_model.predict(X_test)

# Gradient Boosting Classifier
gb_model = GradientBoostingClassifier(n_estimators=100, random_state = 42, verbose=0)
gb_model.fit(X_train, y_train, sample_weight=None)
gb_predictions = gb_model.predict(X_test)
# Add a new Dataframe to see the results clearly
test_results_df = X_test.copy()
test_results_df['True_Risk_Level (Random Forest)'] = y_test
test_results_df['Predicted_Risk_level (Random Forest)'] = rf_predictions
test_results_df['True_Risk_Level (Gradient Boosting)'] = y_test
test_results_df['Predicted_Risk_Level (Gradient Boosting)'] = gb_predictions

test_results_df.head(50)

X_train: (7774, 8), X_test: (1944, 8), y_train: (7774,), y_test: (1944,)


,track,scan,bright_ti5,bright_ti4,latitude,longitude,is_daytime,confidence_num,True_Risk_Level (Random Forest),Predicted_Risk_level (Random Forest),True_Risk_Level (Gradient Boosting),Predicted_Risk_Level (Gradient Boosting)
5909,0.39,0.47,314.22,343.77,31.63309,5.98056,1,0.0,2,0,2,1
4348,0.40,0.48,292.26,307.37,-16.16359,-58.47479,0,0.0,0,0,0,0
1111,0.63,0.46,280.02,326.28,66.68375,77.31839,1,0.0,0,0,0,0
2055,0.55,0.33,271.42,304.98,51.43451,134.83208,0,0.0,0,0,0,0
5143,0.57,0.35,283.56,298.93,31.99234,12.94554,0,0.0,0,0,0,0
7065,0.43,0.38,315.19,341.33,35.87790,4.44963,1,0.0,0,0,0,0
2344,0.48,0.47,280.49,320.23,57.42126,60.08202,0,0.0,0,0,0,0
1488,0.58,0.37,274.27,295.62,64.35340,77.82371,0,0.0,0,0,0,0
1175,0.40,0.47,285.12,330.23,66.12799,77.31472,1,0.0,0,0,0,0
6336,0.37,0.40,291.66,308.03,30.87724,5.64057,0,0.0,1,0,1,0


Accuracy Score: How good were the model's guesses?

In [104]:
from sklearn.metrics import accuracy_score as acc
from sklearn.metrics import precision_score as pre
from sklearn.metrics import recall_score as rc
from sklearn.metrics import f1_score as f1
from sklearn.metrics import classification_report as cl_report
rf_acc_score = acc(test_results_df['True_Risk_Level (Random Forest)'], test_results_df['Predicted_Risk_level (Random Forest)'])
rf_pre_score = pre(test_results_df['True_Risk_Level (Random Forest)'], test_results_df['Predicted_Risk_level (Random Forest)'], average='weighted')
rf_rec_score = rc(test_results_df['True_Risk_Level (Random Forest)'], test_results_df['Predicted_Risk_level (Random Forest)'], average='weighted')
rf_f1_score = f1(test_results_df['True_Risk_Level (Random Forest)'], test_results_df['Predicted_Risk_level (Random Forest)'], average='weighted')

gb_acc_score = acc(test_results_df['True_Risk_Level (Gradient Boosting)'], test_results_df['Predicted_Risk_Level (Gradient Boosting)'])
gb_pre_score = pre( test_results_df['True_Risk_Level (Gradient Boosting)'], test_results_df['Predicted_Risk_Level (Gradient Boosting)'], average='weighted')
gb_rec_score = rc(test_results_df['True_Risk_Level (Gradient Boosting)'], test_results_df['Predicted_Risk_Level (Gradient Boosting)'], average='weighted')
gb_f1_score = f1(test_results_df['True_Risk_Level (Gradient Boosting)'], test_results_df['Predicted_Risk_Level (Gradient Boosting)'], average='weighted')
print(f" Accuracy Score: Random Forest --> {rf_acc_score}\n Precision Score: Random Forest --> {rf_pre_score}\n Recall Score: Random Forest --> {rf_rec_score}\n F1 Score: Random Forest --> {rf_f1_score}")
print("\n")
print(f" Accuracy Score: Gradient Boosting --> {gb_acc_score}\n Precision Score: Gradient Boosting --> {gb_pre_score}\n Recall Score: Gradient Boosting --> {gb_rec_score}\n F1 Score: Gradient Boosting --> {gb_f1_score}")
print(cl_report(
    test_results_df['True_Risk_Level (Gradient Boosting)'], 
    test_results_df['Predicted_Risk_Level (Gradient Boosting)'], 
    target_names=['Low Risk (0)', 'Moderate Risk (1)', 'High Risk (2)', 'Extreme Risk (3)']
))


 Accuracy Score: Random Forest --> 0.8132716049382716
 Precision Score: Random Forest --> 0.806644599592214
 Recall Score: Random Forest --> 0.8132716049382716
 F1 Score: Random Forest --> 0.8006411430781178


 Accuracy Score: Gradient Boosting --> 0.8065843621399177
 Precision Score: Gradient Boosting --> 0.7932586704774457
 Recall Score: Gradient Boosting --> 0.8065843621399177
 F1 Score: Gradient Boosting --> 0.7903246912490365
                   precision    recall  f1-score   support

     Low Risk (0)       0.87      0.94      0.90      1386
Moderate Risk (1)       0.60      0.58      0.59       405
    High Risk (2)       0.68      0.20      0.30       117
 Extreme Risk (3)       0.50      0.28      0.36        36

         accuracy                           0.81      1944
        macro avg       0.66      0.50      0.54      1944
     weighted avg       0.79      0.81      0.79      1944



Prediction: Only 83-85% F1 Score, let's test XG Gradient Boosting

In [105]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV

sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
xgb_model = XGBClassifier(objective='multi:softprob',
    num_class=4,
    learning_rate=0.15,     
    max_depth=7,            
    min_child_weight=1,   
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42)
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
xgb_preds = xgb_model.predict(X_test)
test_results_df['True_Risk_Level (XG Gradient Boosting)'] = y_test
test_results_df['Predicted_Risk_Level (XG Gradient Boosting)'] = xgb_preds
xgb_acc_score = acc(test_results_df['True_Risk_Level (XG Gradient Boosting)'], test_results_df['Predicted_Risk_Level (XG Gradient Boosting)'])
xgb_pre_score = pre( test_results_df['True_Risk_Level (XG Gradient Boosting)'], test_results_df['Predicted_Risk_Level (XG Gradient Boosting)'], average='weighted')
xgb_rec_score = rc(test_results_df['True_Risk_Level (XG Gradient Boosting)'], test_results_df['Predicted_Risk_Level (XG Gradient Boosting)'], average='weighted')
xgb_f1_score = f1(test_results_df['True_Risk_Level (XG Gradient Boosting)'], test_results_df['Predicted_Risk_Level (XG Gradient Boosting)'], average='weighted')
print(f" Accuracy Score: XG Gradient Boosting --> {xgb_acc_score}\n Precision Score: XG Gradient Boosting--> {xgb_pre_score}\n Recall Score: XG Gradient Boosting --> {xgb_rec_score}\n F1 Score: XG Gradient Boosting --> {xgb_f1_score}")
print(classification_report(y_test, xgb_preds, target_names=['Low', 'Moderate', 'High', 'Extreme']))

# The results weren't as I expected, so I will now try GridSearch to find the best settings
xgb_param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_base = XGBClassifier(
    objective='multi:softprob',
    num_class=4,
    random_state=42
)

xgb_grid = GridSearchCV(
    estimator=xgb_base,
    param_grid=xgb_param_grid,
    scoring='f1_macro',
    cv=3,
    verbose=1,
    n_jobs=-1
)

xgb_grid.fit(X_train, y_train, sample_weight=sample_weights)
best_xgb = xgb_grid.best_estimator_
print(f"\nWinning XGBoost Settings: {xgb_grid.best_params_}")

tuned_xgb_preds = best_xgb.predict(X_test)
print("\nTUNED XGBOOST SCORECARD:")
print(classification_report(y_test, tuned_xgb_preds, target_names=['Low', 'Moderate', 'High', 'Extreme']))

 Accuracy Score: XG Gradient Boosting --> 0.8117283950617284
 Precision Score: XG Gradient Boosting--> 0.8274429850287695
 Recall Score: XG Gradient Boosting --> 0.8117283950617284
 F1 Score: XG Gradient Boosting --> 0.8169528336626302
              precision    recall  f1-score   support

         Low       0.93      0.87      0.90      1386
    Moderate       0.57      0.72      0.64       405
        High       0.59      0.52      0.55       117
     Extreme       0.57      0.47      0.52        36

    accuracy                           0.81      1944
   macro avg       0.66      0.65      0.65      1944
weighted avg       0.83      0.81      0.82      1944

Fitting 3 folds for each of 36 candidates, totalling 108 fits

Winning XGBoost Settings: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 4, 'subsample': 0.8}

TUNED XGBOOST SCORECARD:
              precision    recall  f1-score   support

         Low       0.94      0.82      0.88      1386
    Moderate       0.53